# 06 · Showcase: a Self-Organising Map on a geodesic sphere

A spherical lattice has no borders, so every SOM node has the same neighbourhood (apart from the 12
five-neighbour corners) and there are no edge effects. This notebook uses

* the dome vertices as SOM nodes,
* `get_neighbours_in_distance` for the ring-shaped neighbourhood function,
* `vertex.set_data` to store each node's weight vector,
* a map projection to show the trained map.

The SOM learns random RGB colours, so the result is a smooth colour field on the sphere.

Script version: [`examples/06_spherical_som.py`](../06_spherical_som.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from dome_utils import neighbour_rings, unique_mesh
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome
from mt.geodesicdome.projection.equal_earth import EqualEarth

## Parameters

In [ ]:
FREQ = 8                 # 642 SOM nodes
ITERATIONS = 3000
MAX_RADIUS = 6           # neighbourhood radius (in rings) at the start of training
SEED = 42

## Training

The learning rate decays exponentially from 0.5 to 0.02 and the neighbourhood shrinks from `MAX_RADIUS`
rings to 1. The neighbourhood is a Gaussian of the ring number around the best-matching unit (BMU).
Every 100 samples we record the **roughness** of the map: the mean colour distance between neighbouring
nodes.

In [ ]:
def train_som(freq=FREQ, iterations=ITERATIONS, max_radius=MAX_RADIUS, seed=SEED):
    rng = np.random.default_rng(seed)
    dome = GeodesicDome(freq)
    points, faces, index_map = unique_mesh(dome)
    vertices = dome.get_all_vertices()

    representative = {}                                    # one stored vertex per physical node
    for v in vertices:
        representative.setdefault(index_map[v.id], v)

    edges = np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]])

    def roughness(w):
        return np.linalg.norm(w[edges[:, 0]] - w[edges[:, 1]], axis=1).mean()

    weights = rng.random((len(points), 3))                 # random initial RGB weight per node
    initial = weights.copy()
    samples = rng.random((iterations, 3))                  # training data: random colours
    history = [(0, roughness(weights))]

    for t, x in enumerate(samples):
        progress = t / iterations
        lr = 0.5 * (0.02 / 0.5) ** progress                       # 0.5 -> 0.02
        radius = max(1, round(max_radius * (1 - progress)))       # max_radius -> 1 rings
        sigma = max(radius / 2, 0.5)

        bmu = int(np.argmin(((weights - x) ** 2).sum(axis=1)))    # best matching unit
        weights[bmu] += lr * (x - weights[bmu])
        for r, ring in enumerate(neighbour_rings(dome, representative[bmu], radius), start=1):
            idx = np.unique([index_map[u.id] for u in ring])
            weights[idx] += lr * np.exp(-r * r / (2 * sigma * sigma)) * (x - weights[idx])
        if (t + 1) % 100 == 0:
            history.append((t + 1, roughness(weights)))

    for v in vertices:                                     # store the result on every stored vertex
        v.set_data(weights[index_map[v.id]])
    return dict(dome=dome, points=points, faces=faces, index_map=index_map, vertices=vertices,
                weights=weights, initial=initial, history=np.array(history))


som = train_som()
h = som['history']
print(f"{len(som['points'])} nodes; mean neighbour colour distance: before {h[0, 1]:.3f}  after {h[-1, 1]:.3f}")

## Learning curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(h[:, 0], h[:, 1], color='#2980b9')
ax.set_xlabel('training samples')
ax.set_ylabel('mean neighbour distance')
ax.set_title('the map becomes smooth as it organises')
ax.grid(alpha=0.3)
plt.show()

## The trained map

In [ ]:
def draw_map(ax, dome, vertices, colour_of_vertex_id, title):
    triangles = EqualEarth().build(dome)
    flat = np.array([v.projected_coord for v in vertices])
    ax.add_collection(PolyCollection(flat[triangles], facecolors=colour_of_vertex_id[triangles].mean(axis=1),
                                     edgecolors='none'))
    ax.autoscale(), ax.set_aspect('equal'), ax.set_axis_off(), ax.set_title(title)


def draw_globe(ax, points, faces, node_colours, azim, title):
    ax.add_collection3d(Poly3DCollection(points[faces], facecolors=node_colours[faces].mean(axis=1),
                                         edgecolors='none'))
    ax.set_xlim(-1, 1), ax.set_ylim(-1, 1), ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1)), ax.view_init(elev=15, azim=azim), ax.set_axis_off(), ax.set_title(title)


def draw_som(fig, som, iterations):
    per_vertex_initial = som['initial'][som['index_map']]              # colour for every stored vertex id
    per_vertex_trained = np.array([v.data for v in som['vertices']])   # read back what we stored
    d, p, f, w = som['dome'], som['points'], som['faces'], som['weights']
    draw_map(fig.add_subplot(1, 4, 1), d, som['vertices'], per_vertex_initial, 'before training (Equal Earth)')
    draw_map(fig.add_subplot(1, 4, 2), d, som['vertices'], per_vertex_trained,
             f'after {iterations} samples (Equal Earth)')
    draw_globe(fig.add_subplot(1, 4, 3, projection='3d'), p, f, w, 30, 'trained, front')
    draw_globe(fig.add_subplot(1, 4, 4, projection='3d'), p, f, w, 210, 'trained, back')
    fig.suptitle(f'Spherical SOM with {len(p)} nodes on GeodesicDome(frequency={d.frequency})', fontsize=14)
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.82, wspace=0.05)


fig = plt.figure(figsize=(16, 4.6))
draw_som(fig, som, ITERATIONS)
plt.show()

## Explore the trained SOM in the rotatable map viewer

`colors='data'` colours the map with `vertex.data`, i.e. the weights we stored above. With live figures,
drag the map to rotate the sphere (see notebook 09 for all the controls).

In [ ]:
from IPython.display import display

from mt.geodesicdome.interactive import ProjectionViewer

with plt.ioff():
    viewer = ProjectionViewer(som['dome'], 'Equal Earth', colors='data', edges=False, figsize=(10, 5.5),
                              title='trained spherical SOM: drag to rotate')
som_view = LiveFigure(fig=viewer.fig)
som_view.show() if WIDGET else display(viewer.fig)

### Try it: train your own

Set the size of the map and the training schedule, then press **Run Interact**. A frequency-8 map with
3000 samples trains in a couple of seconds; larger maps and more samples take proportionally longer.

In [ ]:
live = LiveFigure(figsize=(16, 4.6))


@widgets.interact_manual(freq=widgets.IntSlider(value=6, min=2, max=16, description='frequency'),
                         iterations=widgets.IntSlider(value=2000, min=200, max=20000, step=200,
                                                      description='samples'),
                         max_radius=widgets.IntSlider(value=5, min=1, max=12, description='start radius'),
                         seed=widgets.IntSlider(value=0, min=0, max=99, description='seed'))
def retrain(freq, iterations, max_radius, seed):
    s = train_som(freq, iterations, max_radius, seed)
    h = s['history']
    print(f"{len(s['points'])} nodes; roughness {h[0, 1]:.3f} -> {h[-1, 1]:.3f}")
    draw_som(live.clear(), s, iterations)
    live.refresh()


live.show()